In [1]:
import os, sys

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"  # JDK 17 (pyspark 4.x exige Java 17+)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

os.environ["HADOOP_HOME"] = r"C:\hadoop"  # winutils.exe/hadoop.dll para o Spark funcionar com FS local no Windows
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin;" + os.environ["PATH"]

import os
os.environ['SPARK_LOCAL_IP'] = '192.168.15.16'

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, greatest, lit
from pyspark.sql.functions import col, sum, when
from pyspark.sql.functions import col, when, sum as _sum, avg, max as _max, min as _min, stddev, percentile_approx


spark = SparkSession \
    .builder \
    .appName("HomeCredit_Bureau") \
    .master("local[1]") \
    .config("spark.driver.host", "192.168.15.16") \
    .config("spark.driver.bindAddress", "192.168.15.16") \
    .getOrCreate()

print(spark.version)

c:\Users\muril\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


4.2.0


In [2]:
# Cell 1 original (carrega só o bureau.csv):
file_path_bureau = r"C:\Users\muril\OneDrive\Desktop\Fonte\bureau.csv"
dados = spark.read.csv(file_path_bureau, header=True, inferSchema=True)

# NOVO — carrega o book_bureau_balance já pronto (do outro notebook)
file_path_book_balance = r"bureau_balance_agg.parquet"
book_bureau_balance = spark.read.parquet(file_path_book_balance)

# NOVO — cruza aqui, no grão SK_ID_BUREAU, ANTES de qualquer outra coisa
dados = dados.join(book_bureau_balance, on="SK_ID_BUREAU", how="left")

dados.createOrReplaceTempView("dados")
dados.show(5)

# Validação — confirma que não duplicou nada
print("Linhas bureau original:", spark.read.csv(file_path_bureau, header=True, inferSchema=True).count())
print("Linhas após join:", dados.count())

+------------+----------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+---------------+------------------+-----------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+------

Criando as flags de janelas temporais

In [3]:
dados = spark.sql("""

    select
        *,
        case when DAYS_CREDIT >= -3  then 1 else 0 end as flag_ultimos_3_meses,
        case when DAYS_CREDIT >= -6  then 1 else 0 end as flag_ultimos_6_meses,
        case when DAYS_CREDIT >= -9  then 1 else 0 end as flag_ultimos_9_meses,
        case when DAYS_CREDIT >= -12 then 1 else 0 end as flag_ultimos_12_meses,
        case when DAYS_CREDIT >= -18 then 1 else 0 end as flag_ultimos_18_meses,
        case when DAYS_CREDIT >= -24 then 1 else 0 end as flag_ultimos_24_meses,
        case when DAYS_CREDIT >= -36 then 1 else 0 end as flag_ultimos_36_meses
    from dados

""")

dados.createOrReplaceTempView("dados")

In [4]:
dados = spark.sql("""

    select
        *,
        case when CREDIT_CURRENCY = 'currency 1' then 1 else 0 end as flag_status_C1,
        case when CREDIT_CURRENCY = 'currency 2' then 1 else 0 end as flag_status_C2,
        case when CREDIT_CURRENCY = 'currency 3' then 1 else 0 end as flag_status_C3,
        case when CREDIT_CURRENCY = 'currency 4' then 1 else 0 end as flag_status_C4
    from dados
    
""")
dados.createOrReplaceTempView("dados")

In [5]:
dados = spark.sql("""

    select 
    *,
    case when CREDIT_ACTIVE = 'Active' then 1 else 0 end as flag_status_Active,
    case when CREDIT_ACTIVE = 'Closed' then 1 else 0 end as flag_status_Closed,
    case when CREDIT_ACTIVE = 'Sold' then 1 else 0 end as flag_status_Sold,
    case when CREDIT_ACTIVE = 'Bad debt' then 1 else 0 end as flag_status_Bad_debt
    from dados
""")

dados.createOrReplaceTempView("dados")

In [6]:
dados.show(5)

+------------+----------+-------------+---------------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+---------------+------------------+-----------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+--------------------------------------+------

#### Construção de Variáveis com as features do bureau_balance

    - Min
    - Max
    - Média
    - Desvio Padrão
    - Sum

In [7]:
colunas_bureau_balance_numericas = [c for c in dados.columns if c.startswith("QTD_FLAG_STATUS_")]

expressoes_bureau_balance_stats = []

for coluna in colunas_bureau_balance_numericas:
    expressoes_bureau_balance_stats.append(_sum(col(coluna)).alias(f"SUM_{coluna}"))
    expressoes_bureau_balance_stats.append(avg(col(coluna)).alias(f"MEAN_{coluna}"))
    expressoes_bureau_balance_stats.append(_max(col(coluna)).alias(f"MAX_{coluna}"))
    expressoes_bureau_balance_stats.append(_min(col(coluna)).alias(f"MIN_{coluna}"))
    expressoes_bureau_balance_stats.append(stddev(col(coluna)).alias(f"STD_{coluna}"))

expressoes_bureau_balance_stats = tuple(expressoes_bureau_balance_stats)

book_bureau_balance_stats = dados.groupBy("SK_ID_CURR").agg(*expressoes_bureau_balance_stats).orderBy("SK_ID_CURR")

print((book_bureau_balance_stats.count(), len(book_bureau_balance_stats.columns)))
book_bureau_balance_stats.createOrReplaceTempView("df_temp01")
book_bureau_balance_stats.show(5)

(305811, 281)
+----------+------------------------------------------+-------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+-------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+-------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+-------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+------------------------------------------+-------------------------------------------+-----------------------

#### Medidas quantitativas abertas por categorias dentro das janelas temporais

In [8]:
# Colunas de status e de janelas criadas nas células anteriores
colunas_agregacao_total_currency = ['flag_status_C1', 'flag_status_C2', 'flag_status_C3', 'flag_status_C4']

colunas_flags = ['flag_ultimos_3_meses', 'flag_ultimos_6_meses', 'flag_ultimos_9_meses',
                  'flag_ultimos_12_meses', 'flag_ultimos_18_meses', 'flag_ultimos_24_meses',
                  'flag_ultimos_36_meses']

expressoes_agregacao_currency = []

for flag in colunas_flags:
    for coluna in colunas_agregacao_total_currency:
        expressoes_agregacao_currency.append(
            sum(when(col(flag) == 1, col(coluna)).otherwise(0)).alias(f"QTD_{coluna.upper()}_{flag.upper()}")
        )

expressoes_agregacao_currency = tuple(expressoes_agregacao_currency)

# Aplicar as expressões de agregação
dados_qtd_currency = dados.groupBy("SK_ID_CURR").agg(*expressoes_agregacao_currency).orderBy("SK_ID_CURR")

print((dados_qtd_currency.count(), len(dados_qtd_currency.columns)))
dados_qtd_currency.createOrReplaceTempView("df_temp02")
dados_qtd_currency.show(5)

(305811, 29)
+----------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+---------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+----------------------------------------+---

In [9]:
# Colunas de status e de janelas criadas nas células anteriores
colunas_agregacao_total_active = ['flag_status_Active', 'flag_status_Closed', 
                                    'flag_status_Sold', 'flag_status_Bad_debt']

expressoes_agregacao_active = []

for flag in colunas_flags:
    for coluna in colunas_agregacao_total_active:
        expressoes_agregacao_active.append(
            sum(when(col(flag) == 1, col(coluna)).otherwise(0)).alias(f"QTD_{coluna.upper()}_{flag.upper()}")
        )

expressoes_agregacao_active = tuple(expressoes_agregacao_active)

# Aplicar as expressões de agregação
dados_qtd_active = dados.groupBy("SK_ID_CURR").agg(*expressoes_agregacao_active).orderBy("SK_ID_CURR")

print((dados_qtd_active.count(), len(dados_qtd_active.columns)))
dados_qtd_active.createOrReplaceTempView("df_temp03")
dados_qtd_active.show(5)

(305811, 29)
+----------+-------------------------------------------+-------------------------------------------+-----------------------------------------+---------------------------------------------+-------------------------------------------+-------------------------------------------+-----------------------------------------+---------------------------------------------+-------------------------------------------+-------------------------------------------+-----------------------------------------+---------------------------------------------+--------------------------------------------+--------------------------------------------+------------------------------------------+----------------------------------------------+--------------------------------------------+--------------------------------------------+------------------------------------------+----------------------------------------------+--------------------------------------------+------------------------------------------

#### Construção de medidas para colunas monetárias nas Janelas Temporais e Razões entre as medidas nas Janelas

- soma
- média
- max
- min
- desvio padrão

In [10]:
colunas_monetarias = [
    'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
    'AMT_CREDIT_SUM_OVERDUE', 'AMT_CREDIT_MAX_OVERDUE', 'AMT_ANNUITY'
]

In [11]:
expressoes_monetarias = []

for flag in colunas_flags:
    # flag_ultimos_3_meses -> U3 (nome mais curto pra facilitar o cruzamento nas razões depois)
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_monetarias:
        # Fora da janela vira NULL — sum/avg/max/min/stddev ignoram NULL automaticamente,
        # não precisa filtrar com WHERE separado por janela
        base = when(col(flag) == 1, col(coluna))

        expressoes_monetarias.append(_sum(base).alias(f"SUM_{coluna}_{sufixo_janela}"))
        expressoes_monetarias.append(avg(base).alias(f"MEAN_{coluna}_{sufixo_janela}"))
        expressoes_monetarias.append(_max(base).alias(f"MAX_{coluna}_{sufixo_janela}"))
        expressoes_monetarias.append(_min(base).alias(f"MIN_{coluna}_{sufixo_janela}"))
        expressoes_monetarias.append(stddev(base).alias(f"STD_{coluna}_{sufixo_janela}"))
        expressoes_monetarias.append(percentile_approx(base, 0.5).alias(f"MEDIAN_{coluna}_{sufixo_janela}"))

expressoes_monetarias = tuple(expressoes_monetarias)

book_bureau_monetario = dados.groupBy("SK_ID_CURR").agg(*expressoes_monetarias).orderBy("SK_ID_CURR")

print((book_bureau_monetario.count(), len(book_bureau_monetario.columns)))
book_bureau_monetario.createOrReplaceTempView("df_temp04")
book_bureau_monetario.show(5)

(305811, 253)
+----------+---------------------+----------------------+---------------------+---------------------+---------------------+------------------------+--------------------------+---------------------------+--------------------------+--------------------------+--------------------------+-----------------------------+---------------------------+----------------------------+---------------------------+---------------------------+---------------------------+------------------------------+-----------------------------+------------------------------+-----------------------------+-----------------------------+-----------------------------+--------------------------------+-----------------------------+------------------------------+-----------------------------+-----------------------------+-----------------------------+--------------------------------+------------------+-------------------+------------------+------------------+------------------+---------------------+--------------

In [12]:
janelas_ordem = ['U3', 'U6', 'U9', 'U12', 'U18', 'U24', 'U36']  # ordem cronológica, do mais recente pro mais antigo

# ---------------------------------------------------------
# Padrão A — U3 (mais recente) vs cada uma das demais janelas
# Mede concentração/aceleração recente frente ao histórico maior
# Gera: U3/U6, U3/U9, U3/U12, U3/U18, U3/U24, U3/U36
# ---------------------------------------------------------
pares_u3_vs_demais = [(janelas_ordem[0], j) for j in janelas_ordem[1:]]
# U3/U6, U3/U9, U3/U12, U3/U18, U3/U24, U3/U36

pares_consecutivos = [(janelas_ordem[i], janelas_ordem[i + 1]) for i in range(1, len(janelas_ordem) - 1)]


def gerar_expressoes_razao(df, pares, colunas, metrica='SUM'):
    """Gera expressões de razão numerador/denominador, só para pares
    onde as duas colunas de fato existem no DataFrame — evita o erro
    de referenciar coluna que não foi criada."""
    colunas_existentes = set(df.columns)
    expressoes = []

    for numerador, denominador in pares:
        for coluna in colunas:
            col_num = f"{metrica}_{coluna}_{numerador}"
            col_den = f"{metrica}_{coluna}_{denominador}"

            if col_num in colunas_existentes and col_den in colunas_existentes:
                expressoes.append(
                    (col(col_num) / when(col(col_den) != 0, col(col_den)))
                    .alias(f"RATIO_{coluna}_{metrica}_{numerador}_{denominador}")
                )
    return expressoes


expressoes_razao_a = gerar_expressoes_razao(book_bureau_monetario, pares_u3_vs_demais, colunas_monetarias, metrica='SUM')
expressoes_razao_b = gerar_expressoes_razao(book_bureau_monetario, pares_consecutivos, colunas_monetarias, metrica='SUM')

book_bureau_razoes = book_bureau_monetario.select(
    "SK_ID_CURR",
    *expressoes_razao_a,
    *expressoes_razao_b
)

print((book_bureau_razoes.count(), len(book_bureau_razoes.columns)))
book_bureau_razoes.createOrReplaceTempView("df_temp05")
book_bureau_razoes.show(5)

(305811, 67)
+----------+------------------------------+-----------------------------------+------------------------------------+--------------------------------------+--------------------------------------+---------------------------+------------------------------+-----------------------------------+------------------------------------+--------------------------------------+--------------------------------------+---------------------------+-------------------------------+------------------------------------+-------------------------------------+---------------------------------------+---------------------------------------+----------------------------+-------------------------------+------------------------------------+-------------------------------------+---------------------------------------+---------------------------------------+----------------------------+-------------------------------+------------------------------------+-------------------------------------+----------------

#### Range e Coeficiente de variação por janela temporal

In [13]:
expressoes_range_cv = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_monetarias:
        base = when(col(flag) == 1, col(coluna))

        # Range = amplitude entre o maior e o menor valor dentro da janela
        expressoes_range_cv.append(
            (_max(base) - _min(base)).alias(f"RANGE_{coluna}_{sufixo_janela}")
        )

        # CV = desvio padrão relativizado pela média — comparável entre clientes de porte diferente
        expressoes_range_cv.append(
            (stddev(base) / when(avg(base) != 0, avg(base))).alias(f"CV_{coluna}_{sufixo_janela}")
        )

expressoes_range_cv = tuple(expressoes_range_cv)

book_bureau_range_cv = dados.groupBy("SK_ID_CURR").agg(*expressoes_range_cv).orderBy("SK_ID_CURR")

print((book_bureau_range_cv.count(), len(book_bureau_range_cv.columns)))
book_bureau_range_cv.createOrReplaceTempView("df_temp06")
book_bureau_range_cv.show(5)

(305811, 85)
+----------+-----------------------+--------------------+----------------------------+-------------------------+-----------------------------+--------------------------+-------------------------------+----------------------------+-------------------------------+----------------------------+--------------------+-----------------+-----------------------+--------------------+----------------------------+-------------------------+-----------------------------+--------------------------+-------------------------------+----------------------------+-------------------------------+----------------------------+--------------------+-----------------+-----------------------+--------------------+----------------------------+-------------------------+-----------------------------+--------------------------+-------------------------------+----------------------------+-------------------------------+----------------------------+--------------------+-----------------+---------------------

#### Crédito e Overdue Ratio - "Foto por Crédito" SK_ID_BUREAU -> SK_ID_CURR

In [14]:
# 1. Calcula a razão em nível de crédito individual (SK_ID_BUREAU)
dados_ratios = dados.withColumn(
    "RATIO_UTILIZACAO_CREDITO",
    col("AMT_CREDIT_SUM_DEBT") / when(col("AMT_CREDIT_SUM") != 0, col("AMT_CREDIT_SUM"))
).withColumn(
    "RATIO_OVERDUE_CREDITO",
    col("AMT_CREDIT_SUM_OVERDUE") / when(col("AMT_CREDIT_SUM") != 0, col("AMT_CREDIT_SUM"))
)

# 2. Agrega por cliente — cada um pode ter vários créditos, então
#    resume com mean/max (o pior caso e a média geral de utilização)
book_bureau_ratios = dados_ratios.groupBy("SK_ID_CURR").agg(
    avg("RATIO_UTILIZACAO_CREDITO").alias("MEAN_RATIO_UTILIZACAO_CREDITO"),
    _max("RATIO_UTILIZACAO_CREDITO").alias("MAX_RATIO_UTILIZACAO_CREDITO"),
    avg("RATIO_OVERDUE_CREDITO").alias("MEAN_RATIO_OVERDUE_CREDITO"),
    _max("RATIO_OVERDUE_CREDITO").alias("MAX_RATIO_OVERDUE_CREDITO")
).orderBy("SK_ID_CURR")

print((book_bureau_ratios.count(), len(book_bureau_ratios.columns)))
book_bureau_ratios.createOrReplaceTempView("df_temp07")
book_bureau_ratios.show(5)

(305811, 5)
+----------+-----------------------------+----------------------------+--------------------------+-------------------------+
|SK_ID_CURR|MEAN_RATIO_UTILIZACAO_CREDITO|MAX_RATIO_UTILIZACAO_CREDITO|MEAN_RATIO_OVERDUE_CREDITO|MAX_RATIO_OVERDUE_CREDITO|
+----------+-----------------------------+----------------------------+--------------------------+-------------------------+
|    100001|           0.2825178450008114|          0.9874047619047619|                       0.0|                      0.0|
|    100002|                     0.136545|                     0.54618|                       0.0|                      0.0|
|    100003|                          0.0|                         0.0|                       0.0|                      0.0|
|    100004|                          0.0|                         0.0|                       0.0|                      0.0|
|    100005|           0.6012561177614977|          0.9547943037974683|                       0.0|               

### Estatísticas das colunas temporais (DAYS_*) por janela

In [15]:
colunas_dias = ['DAYS_CREDIT', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT',
                 'DAYS_CREDIT_UPDATE', 'CREDIT_DAY_OVERDUE']

expressoes_dias = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_dias:
        base = when(col(flag) == 1, col(coluna))

        expressoes_dias.append(avg(base).alias(f"MEAN_{coluna}_{sufixo_janela}"))
        expressoes_dias.append(percentile_approx(base, 0.5).alias(f"MEDIAN_{coluna}_{sufixo_janela}"))
        expressoes_dias.append(_max(base).alias(f"MAX_{coluna}_{sufixo_janela}"))
        expressoes_dias.append(_min(base).alias(f"MIN_{coluna}_{sufixo_janela}"))
        expressoes_dias.append(stddev(base).alias(f"STD_{coluna}_{sufixo_janela}"))

expressoes_dias = tuple(expressoes_dias)

book_bureau_dias = dados.groupBy("SK_ID_CURR").agg(*expressoes_dias).orderBy("SK_ID_CURR")

print((book_bureau_dias.count(), len(book_bureau_dias.columns)))
book_bureau_dias.createOrReplaceTempView("df_temp08")
book_bureau_dias.show(5)

(305811, 176)
+----------+-------------------+---------------------+------------------+------------------+------------------+---------------------------+-----------------------------+--------------------------+--------------------------+--------------------------+-------------------------+---------------------------+------------------------+------------------------+------------------------+--------------------------+----------------------------+-------------------------+-------------------------+-------------------------+--------------------------+----------------------------+-------------------------+-------------------------+-------------------------+-------------------+---------------------+------------------+------------------+------------------+---------------------------+-----------------------------+--------------------------+--------------------------+--------------------------+-------------------------+---------------------------+------------------------+----------------------

### CNT_CREDIT_PROLONG por janela (é contador, tratamento próprio)

In [16]:
expressoes_prolong = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')
    base = when(col(flag) == 1, col("CNT_CREDIT_PROLONG"))

    expressoes_prolong.append(_sum(base).alias(f"SUM_CNT_CREDIT_PROLONG_{sufixo_janela}"))
    expressoes_prolong.append(_max(base).alias(f"MAX_CNT_CREDIT_PROLONG_{sufixo_janela}"))
    expressoes_prolong.append(avg(base).alias(f"MEAN_CNT_CREDIT_PROLONG_{sufixo_janela}"))

expressoes_prolong = tuple(expressoes_prolong)

book_bureau_prolong = dados.groupBy("SK_ID_CURR").agg(*expressoes_prolong).orderBy("SK_ID_CURR")

print((book_bureau_prolong.count(), len(book_bureau_prolong.columns)))
book_bureau_prolong.createOrReplaceTempView("df_temp09")
book_bureau_prolong.show(5)

(305811, 22)
+----------+-------------------------+-------------------------+--------------------------+-------------------------+-------------------------+--------------------------+-------------------------+-------------------------+--------------------------+--------------------------+--------------------------+---------------------------+--------------------------+--------------------------+---------------------------+--------------------------+--------------------------+---------------------------+--------------------------+--------------------------+---------------------------+
|SK_ID_CURR|SUM_CNT_CREDIT_PROLONG_U3|MAX_CNT_CREDIT_PROLONG_U3|MEAN_CNT_CREDIT_PROLONG_U3|SUM_CNT_CREDIT_PROLONG_U6|MAX_CNT_CREDIT_PROLONG_U6|MEAN_CNT_CREDIT_PROLONG_U6|SUM_CNT_CREDIT_PROLONG_U9|MAX_CNT_CREDIT_PROLONG_U9|MEAN_CNT_CREDIT_PROLONG_U9|SUM_CNT_CREDIT_PROLONG_U12|MAX_CNT_CREDIT_PROLONG_U12|MEAN_CNT_CREDIT_PROLONG_U12|SUM_CNT_CREDIT_PROLONG_U18|MAX_CNT_CREDIT_PROLONG_U18|MEAN_CNT_CREDIT_PROLONG_

### Curtose das colunas monetárias por janela

In [17]:
from pyspark.sql.functions import kurtosis

expressoes_kurtosis = []

for flag in colunas_flags:
    sufixo_janela = flag.replace('flag_ultimos_', 'U').replace('_meses', '')

    for coluna in colunas_monetarias:
        base = when(col(flag) == 1, col(coluna))
        expressoes_kurtosis.append(kurtosis(base).alias(f"KURTOSIS_{coluna}_{sufixo_janela}"))

expressoes_kurtosis = tuple(expressoes_kurtosis)

book_bureau_kurtosis = dados.groupBy("SK_ID_CURR").agg(*expressoes_kurtosis).orderBy("SK_ID_CURR")

print((book_bureau_kurtosis.count(), len(book_bureau_kurtosis.columns)))
book_bureau_kurtosis.createOrReplaceTempView("df_temp10")
book_bureau_kurtosis.show(5)

(305811, 43)
+----------+--------------------------+-------------------------------+--------------------------------+----------------------------------+----------------------------------+-----------------------+--------------------------+-------------------------------+--------------------------------+----------------------------------+----------------------------------+-----------------------+--------------------------+-------------------------------+--------------------------------+----------------------------------+----------------------------------+-----------------------+---------------------------+--------------------------------+---------------------------------+-----------------------------------+-----------------------------------+------------------------+---------------------------+--------------------------------+---------------------------------+-----------------------------------+-----------------------------------+------------------------+---------------------------+-----

### Razões em janelas temporais para as colunas categóricas

In [18]:
def gerar_expressoes_razao_categorica(df, pares_flags, colunas):
    """Mesma lógica de gerar_expressoes_razao, mas usa o padrão de nome
    das colunas categóricas: QTD_{coluna}_{FLAG_ULTIMOS_N_MESES}"""
    colunas_existentes = set(df.columns)
    expressoes = []

    for flag_num, flag_den in pares_flags:
        sufixo_num = flag_num.replace('flag_ultimos_', 'U').replace('_meses', '')
        sufixo_den = flag_den.replace('flag_ultimos_', 'U').replace('_meses', '')

        for coluna in colunas:
            col_num = f"QTD_{coluna.upper()}_{flag_num.upper()}"
            col_den = f"QTD_{coluna.upper()}_{flag_den.upper()}"

            if col_num in colunas_existentes and col_den in colunas_existentes:
                expressoes.append(
                    (col(col_num) / when(col(col_den) != 0, col(col_den)))
                    .alias(f"RATIO_{coluna}_{sufixo_num}_{sufixo_den}")
                )
    return expressoes


# Pares de FLAGS (nome completo), no mesmo padrão A/B que você já usa
pares_flags_u3_vs_demais = [(colunas_flags[0], f) for f in colunas_flags[1:]]
pares_flags_consecutivos = [(colunas_flags[i], colunas_flags[i + 1]) for i in range(1, len(colunas_flags) - 1)]

expressoes_razao_currency = (
    gerar_expressoes_razao_categorica(dados_qtd_currency, pares_flags_u3_vs_demais, colunas_agregacao_total_currency) +
    gerar_expressoes_razao_categorica(dados_qtd_currency, pares_flags_consecutivos, colunas_agregacao_total_currency)
)

expressoes_razao_active = (
    gerar_expressoes_razao_categorica(dados_qtd_active, pares_flags_u3_vs_demais, colunas_agregacao_total_active) +
    gerar_expressoes_razao_categorica(dados_qtd_active, pares_flags_consecutivos, colunas_agregacao_total_active)
)

book_bureau_razoes_categoricas = (
    dados_qtd_currency.select("SK_ID_CURR", *expressoes_razao_currency)
    .join(dados_qtd_active.select("SK_ID_CURR", *expressoes_razao_active), on="SK_ID_CURR", how="left")
)

print((book_bureau_razoes_categoricas.count(), len(book_bureau_razoes_categoricas.columns)))
book_bureau_razoes_categoricas.createOrReplaceTempView("df_temp11")
book_bureau_razoes_categoricas.show(5)

(305811, 89)
+----------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+---------------------------+--------------------------+--------------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+---------------------------+----------------------------+----------------------------+----------------------------+----

### Aceleração (variação da tendência entre janelas consecutivas)

In [19]:
# Passo 1 — gera razões passo-a-passo (todas consecutivas, incluindo U3/U6),
# com prefixo diferente de RATIO_ pra não colidir com o df_temp05 existente
pares_step = [(janelas_ordem[i], janelas_ordem[i + 1]) for i in range(len(janelas_ordem) - 1)]
# U3/U6, U6/U9, U9/U12, U12/U18, U18/U24, U24/U36

def gerar_expressoes_step(df, pares, colunas, metrica='SUM'):
    colunas_existentes = set(df.columns)
    expressoes = []
    for numerador, denominador in pares:
        for coluna in colunas:
            col_num = f"{metrica}_{coluna}_{numerador}"
            col_den = f"{metrica}_{coluna}_{denominador}"
            if col_num in colunas_existentes and col_den in colunas_existentes:
                expressoes.append(
                    (col(col_num) / when(col(col_den) != 0, col(col_den)))
                    .alias(f"STEP_{coluna}_{metrica}_{numerador}_{denominador}")
                )
    return expressoes

expressoes_step = gerar_expressoes_step(book_bureau_monetario, pares_step, colunas_monetarias, metrica='SUM')

book_bureau_step = book_bureau_monetario.select("SK_ID_CURR", *expressoes_step)

# Passo 2 — aceleração = diferença entre dois STEPs consecutivos
expressoes_aceleracao = []
colunas_step_existentes = set(book_bureau_step.columns)

for coluna in colunas_monetarias:
    for i in range(len(pares_step) - 1):
        num1, den1 = pares_step[i]
        num2, den2 = pares_step[i + 1]

        col_step1 = f"STEP_{coluna}_SUM_{num1}_{den1}"
        col_step2 = f"STEP_{coluna}_SUM_{num2}_{den2}"

        if col_step1 in colunas_step_existentes and col_step2 in colunas_step_existentes:
            expressoes_aceleracao.append(
                (col(col_step2) - col(col_step1)).alias(f"ACELERACAO_{coluna}_{num1}_{den1}_vs_{num2}_{den2}")
            )

book_bureau_aceleracao = book_bureau_step.select("SK_ID_CURR", *expressoes_aceleracao)

print((book_bureau_aceleracao.count(), len(book_bureau_aceleracao.columns)))
book_bureau_aceleracao.createOrReplaceTempView("df_temp12")
book_bureau_aceleracao.show(5)

(305811, 31)
+----------+----------------------------------------+-----------------------------------------+-------------------------------------------+--------------------------------------------+--------------------------------------------+---------------------------------------------+----------------------------------------------+------------------------------------------------+-------------------------------------------------+-------------------------------------------------+----------------------------------------------+-----------------------------------------------+-------------------------------------------------+--------------------------------------------------+--------------------------------------------------+------------------------------------------------+-------------------------------------------------+---------------------------------------------------+----------------------------------------------------+----------------------------------------------------+------------

### Centro Massa

In [20]:
colunas_peso = ['AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM']

expressoes_centro_massa = []

for coluna_peso in colunas_peso:
    expressoes_centro_massa.append(
        (
            _sum(col("DAYS_CREDIT") * col(coluna_peso)) /
            when(_sum(col(coluna_peso)) != 0, _sum(col(coluna_peso)))
        ).alias(f"CENTRO_MASSA_DAYS_CREDIT_POR_{coluna_peso}")
    )

expressoes_centro_massa = tuple(expressoes_centro_massa)

book_bureau_centro_massa = dados.groupBy("SK_ID_CURR").agg(*expressoes_centro_massa).orderBy("SK_ID_CURR")

print((book_bureau_centro_massa.count(), len(book_bureau_centro_massa.columns)))
book_bureau_centro_massa.createOrReplaceTempView("df_temp13")
book_bureau_centro_massa.show(5)

(305811, 3)
+----------+------------------------------------------------+-------------------------------------------+
|SK_ID_CURR|CENTRO_MASSA_DAYS_CREDIT_POR_AMT_CREDIT_SUM_DEBT|CENTRO_MASSA_DAYS_CREDIT_POR_AMT_CREDIT_SUM|
+----------+------------------------------------------------+-------------------------------------------+
|    100001|                             -195.81234869567186|                         -568.8684397931696|
|    100002|                                         -1042.0|                        -1019.2421722586109|
|    100003|                                            NULL|                         -775.2591191964226|
|    100004|                                            NULL|                         -866.9082183563288|
|    100005|                             -133.65889496726385|                        -154.60552770701509|
+----------+------------------------------------------------+-------------------------------------------+
only showing top 5 rows


## Join Final

In [21]:
book_bureau = spark.sql("""

    select
        d1.SK_ID_CURR,
        d1.*EXCEPT (SK_ID_CURR),
        d2.*EXCEPT (SK_ID_CURR),
        d3.*EXCEPT (SK_ID_CURR),
        d4.*EXCEPT (SK_ID_CURR),
        d5.*EXCEPT (SK_ID_CURR),
        d6.*EXCEPT (SK_ID_CURR),
        d7.*EXCEPT (SK_ID_CURR),
        d8.*EXCEPT (SK_ID_CURR),
        d9.*EXCEPT (SK_ID_CURR),
        d10.*EXCEPT (SK_ID_CURR),
        d11.*EXCEPT (SK_ID_CURR),
        d12.*EXCEPT (SK_ID_CURR),
        d13.*EXCEPT (SK_ID_CURR)
    from df_temp01 as d1
    left join df_temp02 as d2 on d1.SK_ID_CURR = d2.SK_ID_CURR
    left join df_temp03 as d3 on d1.SK_ID_CURR = d3.SK_ID_CURR
    left join df_temp04 as d4 on d1.SK_ID_CURR = d4.SK_ID_CURR
    left join df_temp05 as d5 on d1.SK_ID_CURR = d5.SK_ID_CURR
    left join df_temp06 as d6 on d1.SK_ID_CURR = d6.SK_ID_CURR
    left join df_temp07 as d7 on d1.SK_ID_CURR = d7.SK_ID_CURR
    left join df_temp08 as d8 on d1.SK_ID_CURR = d8.SK_ID_CURR
    left join df_temp09 as d9 on d1.SK_ID_CURR = d9.SK_ID_CURR
    left join df_temp10 as d10 on d1.SK_ID_CURR = d10.SK_ID_CURR
    left join df_temp11 as d11 on d1.SK_ID_CURR = d11.SK_ID_CURR
    left join df_temp12 as d12 on d1.SK_ID_CURR = d12.SK_ID_CURR
    left join df_temp13 as d13 on d1.SK_ID_CURR = d13.SK_ID_CURR

""")

print((book_bureau.count(), len(book_bureau.columns)))

# Checagem de duplicadas — sempre rodar depois de mexer no join
colunas = book_bureau.columns
duplicadas = [c for c in colunas if colunas.count(c) > 1]
print(set(duplicadas))

(305811, 1101)
set()


In [ ]:
df_temp_bureau = book_bureau.repartition(1)
df_temp_bureau.write.mode("overwrite").parquet("bureau_agg.parquet")